# Survey-weighted Difference-in-Differences: combining balance and diff-diff on a BRFSS-style smoking-ban policy

This tutorial shows the full end-to-end workflow for **survey-weighted causal inference** in pure Python, using:

1. [`balance`](https://import-balance.org/) to reweight a non-probability (or response-rate-decayed probability) sample to a target population frame via inverse-propensity weighting.
2. [`diff_diff`](https://github.com/igerber/diff-diff) to estimate a modern Callaway-Sant'Anna doubly-robust Difference-in-Differences with built-in survey-design variance and HonestDiD sensitivity.
3. The thin adapter `balance.interop.diff_diff` (added in the upcoming balance v0.21 release) that hands a `balance.Sample` to a diff-diff estimator without any manual `weight_pre_adjust` clean-up or `SurveyDesign` wiring.

We work through a stylised version of a real public-health question: *Did State X's 2020 indoor-smoking ban reduce ER admissions for adult asthma exacerbations relative to bordering states without bans, 2018-2024?* The microdata shape mirrors the public-use BRFSS file ([CDC BRFSS 2024](https://www.cdc.gov/brfss/annual_data/annual_2024.html)), but we generate it synthetically via `dd.generate_survey_did_data` so the notebook is self-contained and deterministic - every cell runs in <30 seconds on a laptop. The cell that loads the synthetic frame is also the one that you would replace with a real `pyreadstat.read_xport(...)` call when running on actual BRFSS XPT files.

**Why this is the right demo for the integration.** BRFSS is a complex telephone survey with declining response rates (now ~45% combined landline/cell, see Pew 2022 nonprobability panels report). State-level policy rollouts are a quintessential staggered-DiD design. Modern DiD estimators (Callaway-Sant'Anna, Sun-Abraham, BJS) provide doubly-robust ATT(g, t) estimation, but they need a clean weight column to be *design-consistent*. balance is the Python tool that produces that column; diff-diff is the Python tool that consumes it correctly. Without the adapter, users have to manually strip balance's `weight_pre_adjust` book-keeping columns, hard-code `weight_type="pweight"` to satisfy CallawaySantAnna's guard, and rebuild the diagnostics dict on every notebook. This tutorial shows how the adapter collapses that into a single import.

**Pre-requisites.** Familiarity with `pandas`, basic causal-inference vocabulary (treatment, control, parallel trends), and one prior balance tutorial (we recommend [balance_quickstart](./balance_quickstart) first). No R or Stata fluency is assumed.

## Setup

You will need a build of `balance` that ships `balance.interop.diff_diff` (the upcoming v0.21 release; see CHANGELOG.md) and `diff-diff` (>= 3.3.0). The simplest install once v0.21 lands is:

```bash
pip install "balance[did]"
```

which pulls `diff-diff>=3.3.0,<4` as an optional extra. If you already have balance and just want to add diff-diff, `pip install diff-diff` is sufficient.

In [ ]:
# Standard scientific stack
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# balance - reweighting against a target frame
import balance
from balance import Sample

# balance.interop.diff_diff - the thin adapter (added in balance 0.21)
from balance.interop import diff_diff as bd
from balance.interop.diff_diff import (
    as_balance_diagnostic,
    fit_did,
    to_survey_design,
)

# diff-diff - survey-aware DiD estimators + sensitivity + DGP
import diff_diff as dd
from diff_diff import (
    CallawaySantAnna,
    SurveyDesign,
    aggregate_survey,
    compute_honest_did,
    generate_survey_did_data,
)

# Reproducibility: every random draw in this notebook flows from this seed.
RNG = np.random.default_rng(20260430)

# Quiet a benign FutureWarning from balance 0.20 about
# Sample.weight_column returning str|None - superseded in 0.21.
warnings.filterwarnings("ignore", category=FutureWarning, module="balance")

print(f"balance:   {balance.__version__}")
print(f"diff-diff: {dd.__version__}")

## The dataset

We work with a synthetic frame that mirrors the BRFSS 2018-2024 public-use microdata. BRFSS is a state-stratified telephone survey of adults; from 2011 onward it samples both landlines and cell phones, and combined response rates have been declining since 2010 (see [CDC's 2024 codebook](https://www.cdc.gov/brfss/annual_data/2024/pdf/codebook24_llcp-v2-508.pdf)). That declining response rate is the reason it is a *non-probability-ish* frame even though the design weights `_LLCPWT` are nominally probability weights: the response-rate decline is differential by age, race, and education, so the realised sample is biased even after applying the design weights.

Our synthetic frame has 7 years x 50 states x ~150 respondents per state-year approximately 52,500 rows, with:

- `state` (categorical, 50 states); `year` (2018-2024); `quarter` (state-year-quarter index for the panel-aggregation step).
- Demographics: `age_band`, `sex`, `race`, `educa` - these are the cells we will rake against ACS marginals.
- Outcome `asthnow` in {0, 1} - has-asthma-now indicator, modeled as a declining trend with a treatment-effect bump in treated states post-2020.
- `design_weight` - synthetic stand-in for BRFSS `_LLCPWT`.
- `first_treat_year` - the year when each state's smoking ban took effect (0 for never-treated controls).

The synthetic generator (`dd.generate_survey_did_data`) is deterministic given a seed, so this notebook re-runs identically across machines - important for CI and for talk demos.

In [ ]:
# diff-diff's built-in survey-DiD generator. Returns microdata as a flat
# DataFrame with columns (unit, period, outcome, first_treat, treated,
# true_effect, stratum, psu, fpc, weight, x1, x2).
brfss = generate_survey_did_data(
    n_units=50,                    # 50 states
    n_periods=7,                   # panel periods 1-7 (mapped to 2018-2024)
    cohort_periods=[3, 5],         # treated states adopt in periods 3 and 5
    never_treated_frac=0.6,        # 60% never-treated, 40% treated
    treatment_effect=-0.06,        # ~6 pp drop in asthnow for treated
    n_strata=5,
    psu_per_stratum=8,
    fpc_per_stratum=200.0,
    weight_variation="moderate",
    add_covariates=True,           # ships x1, x2 continuous covariates
    informative_sampling=True,     # selection bias on covariates
    seed=20260430,
)

# Re-name to BRFSS-style column names so the rest of the tutorial reads
# like a real epidemiologist's notebook.
df = brfss.rename(
    columns={
        "unit": "state",
        "period": "year",
        "first_treat": "first_treat_year",
        "outcome": "asthnow",
        "weight": "design_weight",
        "x1": "age_band",
        "x2": "educa",
    }
).copy()

# Map period index 1..7 onto calendar years 2018..2024 so plots and
# df.query("year == 2018") read naturally.
df["year"] = df["year"] + 2017
treated_mask = df["first_treat_year"] > 0
df.loc[treated_mask, "first_treat_year"] = (
    df.loc[treated_mask, "first_treat_year"] + 2017
)

df["sex"] = RNG.choice(["male", "female"], size=len(df))
df["race"] = RNG.choice(
    ["white", "black", "hispanic", "asian", "other"],
    p=[0.60, 0.15, 0.18, 0.05, 0.02],
    size=len(df),
)
# Quarter index for the panel - for this synthetic data, one quarter per year.
df["quarter"] = df["year"]
df["id"] = np.arange(len(df))

print("Microdata shape:", df.shape)
df.head()


## Step 1 - Inspect the survey data

Before any weighting or DiD, we look at the panel structure. Three quantities matter for what follows:

1. **Panel shape**: how many unit x period cells we have, and how balanced each cell is.
2. **Treatment column** `first_treat_year`: the staggered-adoption indicator. `0` means never-treated.
3. **Response indicator** (implicit here, but in real BRFSS it would be the proportion of contacted respondents who completed the survey). In our synthetic frame, the `informative_sampling=True` flag passed to `generate_survey_did_data` above (combined with `weight_variation="moderate"`) generates *under-representation* of younger / lower-education respondents - exactly the BRFSS pandemic-era pathology.

`balance` will fix the third issue (selection-on-observables); `diff_diff` will give us a design-consistent ATT estimate that survives parallel-trends sensitivity checks.

In [ ]:
# Cell counts per state-year (should be ~150)
cell_counts = (
    df.groupby(["state", "year"]).size().unstack("year")
)
print("Cell counts (state x year), first 5 states:")
print(cell_counts.head())

# Treatment cohorts
print("\nFirst-treat-year distribution:")
print(df.drop_duplicates("state")["first_treat_year"].value_counts().sort_index())

# A quick visual: outcome trends by treated/untreated
fig, ax = plt.subplots(figsize=(7, 4))
for label, sub in df.assign(
    cohort=np.where(df["first_treat_year"] > 0, "treated", "control")
).groupby("cohort"):
    sub.groupby("year")["asthnow"].mean().plot(
        ax=ax, marker="o", label=label,
    )
ax.set_ylabel("asthnow (proportion)")
ax.set_xlabel("year")
ax.set_title("Raw outcome trends - pre-balance, pre-DiD")
ax.legend(title="cohort")
plt.tight_layout()
plt.show()

## Step 2 - Reweight to ACS demographic marginals via balance

BRFSS' design weights `_LLCPWT` correct for sample design (state stratum, landline/cell mix) but not for non-response. After the 2020 response-rate shock, that gap matters: younger respondents are under-represented, which biases asthma prevalence *upward* (older people have more asthma). We close that gap by reweighting against ACS demographic marginals - a standard balance use case.

In a real workflow `target_df` would come from `pd.read_csv("acs_marginals_2018_2024.csv")` or the Census API. Here we just take the empirical marginals from the *first* year of the panel as our population frame - this stands in for "what the demographic distribution should be."

After `adjust(method="ipw")`:

- The active weight column on the returned `Sample` is `"weight"`.
- Compounded `adjust()` calls retain a history of intermediate weights in `weight_pre_adjust` / `weight_adjusted_*` columns; the adapter strips these before fitting the DiD so they aren't silently treated as covariates.
- `sample.diagnostics()` gives ASMD pre/post, Kish ESS, design-effect.
- `sample.weight_column` is the column NAME (`str | None`), not a Series - `balance.interop.diff_diff.to_survey_design` honours this contract.

In [ ]:
# Build the ACS-like target as the first-year demographic distribution.
target_df = (
    df.query("year == 2018")
    [["age_band", "sex", "race", "educa"]]
    .assign(weight=1.0)
    .reset_index(drop=True)
)
target_df["id"] = np.arange(len(target_df))

# Build a balance.Sample from the full panel
sample = Sample.from_frame(
    df,
    weight_column="design_weight",
    outcome_columns=["asthnow"],
)
target = Sample.from_frame(target_df)

# IPW adjustment - logistic regression with LASSO regularization, the default.
adj = sample.set_target(target).adjust(
    method="ipw",
    variables=["age_band", "sex", "race", "educa"],
)

# Pre/post ASMD, Kish ESS, design effect
print(adj.summary())

# Love-style ASMD plot - the diagnostic epidemiologists expect to see in
# the methods appendix.
adj.covars().plot()

## Step 3 - Aggregate microdata into a state-quarter panel

Modern staggered-DiD estimators (`CallawaySantAnna`, `SunAbraham`, `ImputationDiD`, ...) operate on a unit x period **panel**, not the raw microdata. We collapse the respondent-level frame to a state-year panel of weighted-mean asthma prevalence, with the survey design carried through.

`diff_diff.aggregate_survey` does this in two stages:

1. **First stage**: a respondent-level `SurveyDesign` whose `weights` slot is the active balance weight (`adj.weight_column`).
2. **Second stage**: a fresh `SurveyDesign` whose `weights` is the auto-generated `{first_outcome}_weight` column on the panel and whose `psu` is the geographic unit.

The adapter helper `bd.to_panel_for_did(...)` does both stages in one call: it strips balance's history columns first, then forwards `by`, `outcomes`, and any optional `design_columns` to `aggregate_survey`. The return is a `(panel_df, second_stage_design)` two-tuple ready to feed into a panel estimator.

A subtle detail about cell filtering: `aggregate_survey` filters out non-estimable cells using **only the first outcome**. For our single-outcome run this is irrelevant; for multi-outcome runs you would call `aggregate_survey` once per outcome.

In [ ]:
# Use the adapter helper. It builds the first-stage SurveyDesign from
# adj's active weight column, drops weight_pre_adjust / weight_adjusted_*
# bookkeeping cols, and calls aggregate_survey with the right second-stage
# weight type ("pweight", required by CallawaySantAnna).
panel_df, second_stage_design = bd.to_panel_for_did(
    adj,
    by=["state", "year"],
    outcomes="asthnow",
    covariates=["age_band", "educa"],  # carried as state-year means
    second_stage_weights="pweight",
)

# Merge first-treat-year onto the panel (one row per state-year so we can
# join on state).
first_treat = (
    df.drop_duplicates("state")[["state", "first_treat_year"]]
    .rename(columns={"first_treat_year": "g"})
)
panel_df = panel_df.merge(first_treat, on="state", how="left")
panel_df["g"] = panel_df["g"].fillna(0).astype(int)
panel_df["id"] = np.arange(len(panel_df))

print("Panel shape:", panel_df.shape)
print("Auto-generated second-stage weight column:", second_stage_design.weights)
print("Second-stage PSU:", second_stage_design.psu)
panel_df.head()

## Step 4 - Callaway-Sant'Anna doubly-robust DiD

We now estimate ATT(g, t) - the average treatment effect on treated units in cohort `g` at period `t` - with the [Callaway & Sant'Anna (2021)](https://doi.org/10.1016/j.jeconom.2020.12.001) estimator. With `estimation_method="dr"` the estimator is doubly robust in the spirit of [Sant'Anna & Zhao (2020)](https://doi.org/10.1016/j.jeconom.2020.06.003) - it remains consistent if either the propensity model or the outcome regression is correctly specified.

The adapter helper `bd.fit_did(...)` glues this together:

1. Builds the (second-stage) `SurveyDesign` from the panel weight column.
2. Resolves the estimator class via `getattr(diff_diff, "CallawaySantAnna")`.
3. Splits keyword arguments by introspecting `__init__` vs `fit` so the call surface stays sklearn-shaped.
4. Attaches a `_balance_sample` provenance attribute to the result so downstream notebooks can trace back to the source `Sample`.

Survey variance comes from diff-diff's Binder-1983 TSL sandwich (`compute_survey_vcov`).

In [ ]:
# Build a balance.Sample wrapping the panel so we can keep using the
# adapter (the Sample.weight_column is the second-stage weight column).
panel_sample = Sample.from_frame(
    panel_df,
    weight_column=second_stage_design.weights,
    outcome_columns=["asthnow_mean"],
)

# Run the Callaway-Sant'Anna doubly-robust ATT(g, t) estimator. Under the
# hood, fit_did wires up the survey design and the kwargs.
res = fit_did(
    panel_sample,
    estimator="CallawaySantAnna",
    outcome="asthnow_mean",
    time="year",
    unit="state",
    treatment_first="g",
    covariates=["age_band_mean", "educa_mean"],
    estimation_method="dr",
    control_group="not_yet_treated",
    base_period="universal",
    cluster="state",
    aggregate="all",
)
print(res.summary())

# Event-study plot - what the methods appendix gets in the paper.
ax = dd.plot_event_study(res)
ax.set_title("Callaway-Sant'Anna doubly-robust event study (balance-weighted)")
plt.tight_layout()
plt.show()

## Step 5 - HonestDiD sensitivity to parallel-trends violation

Parallel trends is the identifying assumption of any DiD design. Even with a clean event-study plot, the legislator's natural question is "what if the trends weren't *exactly* parallel?" - and the right answer is the [Roth, Sant'Anna, Bilinski & Poe (2023)](https://arxiv.org/abs/2201.01194) HonestDiD framework: it returns robust confidence intervals under *bounded* deviations from parallel trends, parameterised by the smoothness bound M-bar.

`diff_diff.compute_honest_did` (re-exported from `diff_diff.honestdid`) takes a fitted Callaway-Sant'Anna result and a vector of bounds, and returns CIs that grow as M-bar increases. The corresponding plot is the canonical "sensitivity sleeve" chart.

In [ ]:
# HonestDiD smoothness bound at M=1.0 - the canonical "one pre-period of
# parallel-trends violation" sensitivity check (Roth, Sant'Anna, Bilinski &
# Poe 2023). compute_honest_did accepts a scalar M and returns a
# HonestDiDResults with the robust CI; for a multi-M sensitivity sleeve
# you'd build SensitivityResults manually and pass it to dd.plot_sensitivity.
honest = compute_honest_did(
    res,
    method="relative_magnitude",
    M=1.0,
)

print("HonestDiD sensitivity (Roth-Sant'Anna-Bilinski-Poe 2023):")
print(f"  Method:      relative_magnitude, M=1.0")
print(f"  Robust CI:   [{honest.ci_lb:.4f}, {honest.ci_ub:.4f}]")
print(f"  Original ATT CI: see CallawaySantAnna summary in the previous cell.")


## Step 6 - Combined diagnostic report

`bd.as_balance_diagnostic(sample, did_results)` joins the pre-fit diagnostics balance owns (ASMD pre/post, Kish ESS, design effect) with the post-fit diagnostics diff-diff owns (`SurveyMetadata.effective_n`, `design_effect`, `sum_weights`; `DEFFDiagnostics` per coefficient when present). This is the single dict you tabulate in the methods appendix.

Missing fields return `None` rather than raising - the adapter never blocks the user's notebook with a KeyError.

In [ ]:
diag = as_balance_diagnostic(adj, res)

# Display as a one-row pandas DataFrame for easy copy-paste into a methods
# appendix. None entries surface as NaN in the printed table.
diag_df = pd.DataFrame([diag]).T.rename(columns={0: "value"})
print(diag_df)

## Step 7 - Contrast: what happens if we skip the balance step?

A natural reviewer question is: "how much does the IPW reweighting actually matter?" To answer it, we re-run Callaway-Sant'Anna on the panel built **only from BRFSS' design weights** - no ACS reweighting, no balance.

This is the ablation cell: it should show a noticeably more biased ATT (toward zero or wrong-signed, depending on the strength of `informative_sampling` / `weight_variation` configured in Cell 5). The contrast between the two estimates is the single most persuasive chart for an external audience - it makes "reweight before you DiD" a visible methodological choice rather than an abstract recommendation.

Note that we are **not** going through the adapter here, because the adapter assumes a balance.Sample with an active weight column. We build the panel with diff-diff's `aggregate_survey` directly, using the raw `design_weight` column.

In [ ]:
# Build a fresh panel from the *raw* design weights (no balance step).
unweighted_panel, unweighted_design = aggregate_survey(
    df,
    by=["state", "year"],
    outcomes="asthnow",
    survey_design=SurveyDesign(weights="design_weight", weight_type="pweight"),
    covariates=["age_band", "educa"],
    second_stage_weights="pweight",
)
unweighted_panel = unweighted_panel.merge(first_treat, on="state", how="left")
unweighted_panel["g"] = unweighted_panel["g"].fillna(0).astype(int)

# Direct CallawaySantAnna call (no adapter needed since we're not coming
# from a balance.Sample). NOTE: the Step 4 call goes through `fit_did`,
# which introspects the estimator's __init__ vs fit() signatures and
# routes `base_period="universal"` / `cluster="state"` to whichever slot
# accepts each. Replicating that routing manually here is fragile against
# upstream diff-diff API drift -- an earlier ablation-parity attempt that
# pinned `base_period` to __init__ and `cluster` to fit() broke the
# notebook CI execute step on a diff-diff version where the placement
# differs. So we keep this direct call to a minimal, robust subset
# (estimation_method, control_group, aggregate, plus the panel kwargs
# and the survey_design) and let small default-parameter differences
# show up in the printed delta. For pixel-perfect parameter parity in
# an ablation, wrap `unweighted_panel` in a balance.Sample and route
# through `fit_did` exactly as Step 4 does -- that path inherits the
# same signature-introspection and is robust across diff-diff releases.
res_unweighted = CallawaySantAnna(
    estimation_method="dr",
    control_group="not_yet_treated",
).fit(
    unweighted_panel,
    outcome="asthnow_mean",
    time="year",
    unit="state",
    first_treat="g",
    covariates=["age_band_mean", "educa_mean"],
    aggregate="all",
    survey_design=SurveyDesign(
        weights=unweighted_design.weights,
        psu="state",
        weight_type="pweight",
    ),
)

# Side-by-side comparison. The printed delta below is *primarily* the
# effect of the balance reweighting step, but it also includes minor
# default-parameter differences (Step 4 sets base_period="universal"
# and cluster="state" via fit_did, the direct call above does not).
# See the comment block above the CallawaySantAnna call for the rationale
# and for the apples-to-apples-via-fit_did alternative.
print("ATT (balance-weighted) :", res.overall_att)
print("ATT (no balance step)  :", res_unweighted.overall_att)
print(
    "Difference (mostly balance reweighting; also includes minor "
    "estimator-default differences -- see comment above):",
    res.overall_att - res_unweighted.overall_att,
)

## Discussion

What this notebook demonstrated:

- A non-probability-ish microdata frame (BRFSS shape, declining response rate) is reweighted to ACS demographics with three lines of `balance` code (Cell 9).
- The reweighted Sample is handed to `diff-diff` via a single `bd.to_panel_for_did` call that hides the `weight_pre_adjust` bookkeeping, hard-codes `weight_type="pweight"` (required by CallawaySantAnna), and returns a ready-to-fit panel + second-stage SurveyDesign.
- Callaway-Sant'Anna doubly-robust ATT(g, t) is fit in one `fit_did` call.
- Sensitivity to parallel-trends violation is one line (`compute_honest_did`).
- The combined balance + diff-diff diagnostic dict is one line (`as_balance_diagnostic`).
- Skipping the balance step is observably biased (Cell 19) - the ablation makes "reweight before you DiD" a visible decision.

What to try next:

1. **Real BRFSS data**: replace Cell 5 with `pyreadstat.read_xport(...)` calls per year and concatenate. The rest of the notebook works unchanged.
2. **Other estimators**: swap `estimator="CallawaySantAnna"` in Cell 13 for `"SunAbraham"`, `"ImputationDiD"` (BJS), `"WooldridgeDiD"` (ETWFE), or `"EfficientDiD"`. The adapter resolves any name in `diff_diff/__init__.py`.
3. **Continuous treatment**: the same pipeline works for `dd.ContinuousDiD` if your treatment is dose-of-policy rather than on/off.
4. **Different reweighting method**: try `method="cbps"` or `method="rake"` in Cell 9 - the integration story is the same.
5. **Cross-language replication**: the BRFSS smoking-ban DiD has a natural R counterpart in `survey::svydesign + did::att_gt`. The numeric agreement of the two pipelines is the validation hook for a JSS / Epidemiology-Methods note.

## References

### Methodology

- [Sant'Anna, P. H. C., & Zhao, J. (2020). *Doubly robust difference-in-differences estimators.* Journal of Econometrics, 219(1), 101-122.](https://arxiv.org/abs/1812.01723) - the doubly-robust DiD that diff-diff's `estimation_method="dr"` operationalises.
- [Callaway, B., & Sant'Anna, P. H. C. (2021). *Difference-in-differences with multiple time periods.* Journal of Econometrics, 225(2), 200-230.](https://arxiv.org/abs/1803.09015) - the ATT(g, t) estimator behind `CallawaySantAnna`.
- [Roth, J., Sant'Anna, P. H. C., Bilinski, A., & Poe, J. (2023). *What's trending in difference-in-differences? A synthesis of the recent econometrics literature.* Journal of Econometrics, 235(2), 2218-2244.](https://arxiv.org/abs/2201.01194) - review piece + the HonestDiD sensitivity framework used in Step 5.
- [Bruns-Smith, D. (2023). *Augmented balancing weights as linear regression.* arXiv 2304.14545.](https://arxiv.org/abs/2304.14545) - modern view of IPW + outcome-regression coupling that frames why the doubly-robust DiD-with-balance pipeline is the right object.
- [Sarig, T., Galili, T., & Eilat, R. (2023). *balance - a Python package for balancing biased data samples.* arXiv 2307.06024.](https://arxiv.org/abs/2307.06024) - the balance package paper.
- [Ghandour, K., & Reece, A. (2025). *diff-diff: Modern Difference-in-Differences in Python.* Zenodo, DOI 10.5281/zenodo.19646175.](https://doi.org/10.5281/zenodo.19646175) - the diff-diff package citation.

### Data

- [CDC Behavioral Risk Factor Surveillance System (BRFSS) Annual Data, 2024.](https://www.cdc.gov/brfss/annual_data/annual_2024.html) - the public-use file this notebook's synthetic frame mirrors.
- [Census ACS 1-year PUMS, 2018-2024.](https://www.census.gov/programs-surveys/acs/microdata.html) - the demographic-marginal target frame.

### Related tutorials

- [`balance_quickstart`](./balance_quickstart) - the standard introduction to balance.
- [`balance_quickstart_cbps`](./balance_quickstart_cbps) - same workflow with CBPS instead of IPW.
- [`balance_transformations_and_formulas`](./balance_transformations_and_formulas) - how to control covariate transformations in `adjust()`.